# Summary

This paper presents a comprehensive analysis of the factors influencing wealth dynamics and social mobility in the United States.

Key findings include:

- **Motivation:** Wealth is unequally distributed, with significant skewness and a thick right tail, where the top 1% holds a disproportionately large share of wealth.
- The lifecycle model developed in the study identifies three main factors driving these outcomes: skewed earnings distribution, differential savings rates across wealth levels, and stochastic idiosyncratic returns to wealth. All three factors are crucial for matching the observed wealth distribution and mobility patterns.
- The model developed matches empirical data well, and counterfactuals provide insights into the relative importance of the above factors in driving wealth accumulation and distribution in the U.S.

For a detailed discussion of the foundational literature, see the [Prior Literature](Benhabib_et_al_2019_prior-literature.ipynb) notebook. For how later work addresses these issues, see the [Subsequent Literature](Benhabib_et_al_2019_subsequent-literature.ipynb) notebook.

## Non-Technical Methodological Overview

The paper develops a macroeconomic model to explore wealth accumulation, distribution, and social mobility in the U.S., focusing on three main forces:

1. **Stochastic Earnings:** Research shows income variability plays a key role in wealth inequality, affecting saving decisions and consumption patterns, particularly for the bottom 60% of households. However, it does not account for the wealth concentration among the wealthiest. The incomplete-markets literature pioneered by {cite:t}`aiyagari1994` established this channel.

2. **Heterogeneous Rate of Return:** Studies have identified significant variations in the risk-adjusted returns on investments across households, contributing to the wealth distribution's long tail. This variation is consistent over time and linked with entrepreneurial activity {cite:p}`quadrini2000, cagetti2006`.

3. **Differential saving rates across wealth levels:** The model includes increasing bequest motives with wealth, suggesting a stronger saving motive among the wealthiest, who aim to leave significant assets for their heirs. This perpetuates the transfer of large estates through generations, as emphasized by {cite:t}`denardi2004`.

The analysis finds that stochastic earnings, differential savings, and capital income risk critically shape the wealth distribution's tail and social mobility. Capital income risk and differential savings widen the wealth distribution's tail and influence mobility, particularly enhancing it at the top end but reducing upward mobility from the bottom 20%. Despite being less impactful in the tail, stochastic earnings are vital for overall wealth mobility. Additionally, their findings suggest a wealth-dependent return rate enhances model fit across the wealth distribution.

## The Model

A fairly simple microfounded model of lifecycle consumption and savings. Each agent's life span is finite and deterministic, T years.

### Features of the model

1. Every period, agents choose how much to consume ($c_t$); end-of-period savings (and so next-period beginning wealth $a_{t+1}$) follow from the budget identity.
2. All agents are subjected to a no-borrowing constraint.
3. Agents leave bequests $a_{T+1}$ at the end of life T.
4. Wealth accumulates from savings and bequests.
5. Each agent is assigned **at birth** (i) a lifetime earnings-profile type $\tau \in \{1, \ldots, 10\}$ drawn from a 10-state intergenerational Markov chain (paper Table 1 gives the ten decile-specific age profiles; Chetty et al. 2014 gives the intergenerational transition), and (ii) a rate-of-return state drawn from a 5-state Markov chain (see "Stochastic Structure" below). **Both $\tau$ and $r$ are constant within a life**, and each is stochastic across generations through its own intergenerational Markov chain, possibly correlated with that of the parent.

### Preferences
Preferences are composed of:
1. per period utility from consumption
2. warm-glow utility from bequests at T, $e(a_{T+1})$
$$
\begin{aligned}
& u(c_t) = \frac{c_{t}^{1 - \sigma}}{1 - \sigma}, \quad \quad e(a_{T+1}) = A\frac{a_{T+1}^{1 - \mu}}{1 - \mu} \\
\end{aligned}
$$

### Recursive Formulation

Given a type assignment $(\tau, r)$ drawn at birth and initial wealth $a_1$, the agent's optimization problem for $t = 1, \ldots, T-1$ is (following the paper's online Appendix A.1 — the authors' authoritative description of the numerical solution):

$$
\begin{aligned}
& V^{\tau, r}_{t}(a_t) \;=\; \max_{c_t} \; u(c_t) + \beta \, V^{\tau, r}_{t+1}(a_{t+1}) \\
& \text{s.t.} \\
& a_{t+1} \;=\; (1+r)\,(a_t - c_t) + w_t(\tau) \qquad \text{(savings earn return; earnings arrive at end of period)} \\
& 0 \;\le\; c_t \;\le\; a_t \qquad \text{(consume out of beginning-of-period wealth; no-borrowing)}
\end{aligned}
$$

with terminal condition

$$
V^{\tau, r}_{T}(a_T) \;=\; \max_{c_T} \; u(c_T) + e(a_{T+1}) \quad \text{subject to} \quad a_{T+1} = (1+r)(a_T - c_T) + w_T(\tau),\; 0 \le c_T \le a_T.
$$

**Notation convention and source.** This formulation follows **online Appendix A.1**. The published paper §I writes the budget compactly as $a' = (1+r)a - c + w$ with constraint $0 \le c \le a$, which is internally inconsistent: the constraint $c \le a$ is genuine under the appendix model, but the budget equation as printed in §I is missing a $(1+r)$ factor on $c$. The Formalized-tier artifacts in this directory ([`bellman-excerpt.md`](bellman-excerpt.md), [`dolo-plus-draft.yaml`](dolo-plus-draft.yaml), [`dynasty-excerpt.md`](dynasty-excerpt.md), [`dolo-plus-dynasty.yaml`](dolo-plus-dynasty.yaml)) build on the appendix model; see [`bellman-excerpt.md`](bellman-excerpt.md) Open Issue #10 for the full audit trail.

For the dolo-plus three-perch decomposition used in `bellman-excerpt.md`, the arrival-perch state is $a_t$ (beginning-of-period wealth), the decision-perch state $m_t$ is **identity** with $a_t$ ($m_t = a_t$ — just a perch label, no transformation), the control is $c_t \in [0, a_t]$, and the continuation-perch state is $a_{t+1}$. Savings $(a_t - c_t)$ earn return $r$, then earnings $w_t(\tau)$ arrive at the end of the period.

**Family structure.** The household problem is a **parameterized family of Bellman problems** indexed by $(\tau, r)$ — ten earnings-profile types and five rate-of-return states, so fifty separate fixed-point problems in the baseline calibration. This is structurally analogous to HAFiscal's $(\beta_i, e)$ type-indexed family (Carroll, Crawley, Frankovic, Tretvoll). Across generations, $(\tau^n, r^n)$ evolve according to the intergenerational Markov chains specified in "Stochastic Structure" below; **within a life, they are fixed parameters of the value function, not state variables.** A dolo-plus YAML must therefore either (a) encode the stage as a parameterized family with $(\tau, r)$ as calibration overrides across stage instances, or (b) encode the dynasty-level structure explicitly with $(\tau, r)$ as discrete states resolved only at birth — HAFiscal uses (a).

### Stochastic Structure

Across generations, the paper specifies:

1. **Rate of return $r^n$** is drawn from a **finite $K$-state Markov chain** with transition $P(r^n \mid r^{n-1})$. In the paper's baseline $K = 5$, with off-diagonal probabilities restricted to decay geometrically away from the diagonal (except the last row, which uses constant off-diagonal probabilities); see paper §I and footnote 13. The full 5×5 matrix is in online Appendix C.1; transcribed in [`dolo-plus-draft.yaml`](dolo-plus-draft.yaml) under `calibration_family.population.Pi_r.matrix`.
2. **Earnings-profile type $\tau^n$** is drawn from a **10-state intergenerational Markov chain**, calibrated from Chetty et al. (2014). The ten profile shapes $\{w_t(\tau)\}_{t=1}^T$ are reported in paper Table 1 (transcribed into [`dolo-plus-draft.yaml`](dolo-plus-draft.yaml) under `calibration_family.by_tau`). Online Appendix B.2 describes the procedure for collapsing Chetty et al.'s 100×100 matrix to 10×10 but does not tabulate the result; the matrix would need separate reconstruction from Chetty et al.'s data tables or from the BBL replication package.
3. $r^n$ and $\tau^n$ are **independent of each other**, but each is **serially correlated across generations**.
4. **Extension (Section IIID):** the paper also considers an extension in which the Markov state space of $r^n$ is allowed to depend on the agent's initial wealth $a_1$, capturing the empirically supported tendency for higher rates of return among the wealthy.

The solution to the above problem is a stochastic difference equation for the initial wealth of dynasties, induced by the $\left\{r^n, \tau^n \right\}_{n}$ processes, mapping $a^{n-1}$ into $\left\{a^{n} \right\}_{n}$, where superscripts correspond to the nth generation:
$$
\begin{aligned}
& a^{n} = g(a^{n-1}; r^{n}, \tau^{n}) \\
\end{aligned}
$$

Under the given assumptions of the model, the following holds:
1. If $\mu  = \sigma$ $\implies$ the stochastic process $\left\{a^{n} \right\}_{n}$ has a stationary distribution with a Pareto right tail
2. If $\mu  < \sigma$ $\implies$ savings rate increases with wealth, $g(\cdot)$ is convex in initial wealth (the rich save proportionally more); a stationary distribution might not exist, but if it does the right tail is at least as thick as Pareto.

The dynasty-level composition — including the lifetime map $g(\cdot)$, the independence of the $\tau$- and $r$-chains, and the paper's stationary-distribution Proposition — is formalized in [`dynasty-excerpt.md`](dynasty-excerpt.md) and [`dolo-plus-dynasty.yaml`](dolo-plus-dynasty.yaml).

### Quantitative Analysis

The paper uses the method of simulated moments (MSM) to identify unknown parameters.

1. Externally calibrate some parameters of the model
2. Estimate remaining parameters of the model by matching the targeted moments generated by the stationary distribution induced by the model and those in the data


## Results

At the estimated parameter values, the model-induced wealth distribution closely resembles the wealth distribution in the data.
-  The estimates point to the existence of differential saving behavior (bequest motives)
- Capital income risk is an important factor in driving wealth inequality

Next, the paper shuts down each of the three main factors listed above. The objective of this counterfactual exercise is to gauge the relative importance of the three mechanisms in driving the distribution of wealth.

### Summary of Counterfactuals

 1. No rate of return heterogeneity $\implies$ higher bequest motive (A doubles relative to baseline) & model can't match the upper tail of the wealth distribution (Table-15-row(3))

 2. No stochastic earnings $\implies$ relative preference for bequests $\uparrow$ while nothing else changes substantially & model does not miss as much in mimicking the upper tail of the wealth distribution $\implies$ stochastic earnings not driving the behavior of the right tail of wealth distribution.
    - However, social mobility matrix fit implied by this counterfactual is bad $\implies$ stochastic earnings matter for social mobility

 3. Homogeneous saving rates $\implies$ preference for bequests $\uparrow$, capital income is riskier & extremely bad fit in matching the upper tail of the wealth distribution

![Counterfactual wealth distributions: model vs. data under alternative assumptions](fig1.png)

Lastly, the paper describes transitional dynamics of the wealth distribution within the confines of the model. In particular, the paper conducted an analysis using the SCF 1962–1963 wealth distribution as a starting point, estimating model parameters to match with the 2007 SCF distribution and previously used transition matrices. The findings highlight a significant rise in wealth inequality during this period, with the top 1% share increasing from 24.2% to 33.6%. The updated estimates reveal that this surge in inequality can be traced through enhanced capital income risk and differential savings, resulting in a skewed wealth distribution that closely matches empirical data, especially at the higher end. However, this model overestimates social mobility across wealth brackets.

![Transitional dynamics of the wealth distribution from SCF 1962–1963 to SCF 2007](fig2.png)

## Conclusion

The authors developed a standard macroeconomic model to explore the distribution of wealth in the United States, with a specific focus on the distribution's tail. The model is notable for its ability to closely fit the observed data across the entire wealth spectrum and accurately capture the social mobility trends. Through their analysis, the authors successfully identified three key factors contributing to wealth accumulation: skewed and persistent earnings distribution, differential saving and bequest rates across wealth levels, and capital income risk associated with entrepreneurship. Each factor plays a distinct and empirically validated role in shaping both the wealth distribution and mobility. The paper also delves into the transitional dynamics of wealth distribution, with preliminary findings suggesting rapid changes over time, indicating promising areas for future research.

## Limitations

The model ignores the following key features that are relevant for a proper quantitative study:
   - Overlapping generation demographic structure, which is crucial for modeling accidental bequests
   - Permanent income heterogeneity and within-lifetime permanent income risk (important for capturing savings done to counter that risk)
   - Luxury-type bequest motives. In their absence, even agents located at the lower end of the wealth distribution save for leaving bequests, but that is not supported by data. Luxury-type bequest motives can solve that problem.
   - Mortality risk, which again alters the saving behavior of retirees
   - Medical risk